In [17]:
# Criado por Marco Garcia - v0.1 - 2026

# 1. Instalações Necessárias
!pip install git+https://github.com/openai/whisper.git -q
!pip install -q -U google-generativeai gTTS

import os
import whisper
import google.generativeai as genai
from gtts import gTTS
from IPython.display import Audio, display, Javascript, HTML
from google.colab import output
from base64 import b64decode

# --- CONFIGURAÇÕES DO AMBIENTE ---
language = "pt"
API_KEY = "INSIRA_SUA_CHAVE_AQUI" # <--- INSIRA SUA CHAVE DO GEMINI AQUI
genai.configure(api_key=API_KEY)

# --- ESTILIZAÇÃO DA INTERFACE (FONTES GRANDES E SEM CORTES) ---
def exibir_card(titulo, conteudo, cor="#d32f2f"):
    display(HTML(f"""
    <div style="margin: 20px 0; padding: 25px; border-left: 10px solid {cor};
                background-color: #ffffff; border-radius: 12px;
                box-shadow: 0 4px 12px rgba(0,0,0,0.15); width: 95%; max-width: 800px;">
        <b style="color: {cor}; font-size: 22px; font-family: sans-serif; text-transform: uppercase; letter-spacing: 1px;">{titulo}</b><br>
        <div style="font-size: 26px; color: #222; font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
                    margin-top: 15px; line-height: 1.4; word-wrap: break-word; white-space: pre-wrap; overflow: visible;">
            {conteudo}
        </div>
    </div>
    """))

# --- INICIALIZAÇÃO DOS MODELOS ---
print('⌛ Carregando o cérebro do Brio...')
model_whisper = whisper.load_model("small")

# Busca automática do modelo disponível
available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
model_choice = "models/gemini-1.5-flash" if "models/gemini-1.5-flash" in available_models else available_models[0]

brio = genai.GenerativeModel(
    model_name=model_choice,
    system_instruction="Seu nome é Brio, assistente de voz do Bra. Responda de forma completa, empática e profissional."
)

# --- GRAVADOR JAVASCRIPT ---
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

# --- FLUXO PRINCIPAL DE EXECUÇÃO ---

try:
    # Título da Aplicação
    display(HTML("<h1 style='color: #d32f2f; font-family: sans-serif; font-size: 42px; margin-bottom: 5px;'>🦁 Brio</h1>"))
    display(HTML("<p style='color: #777; font-family: sans-serif; font-size: 18px; margin-top: 0;'>Assistente de Voz Inteligente | Bradesco Bootcamp</p>"))
    display(HTML("<hr style='border: 0; height: 2px; background: #eee; margin-bottom: 30px;'>"))

    # Passo 1: Ouvir
    exibir_card("Status", "🎙️ O Brio está ouvindo... Fale agora!", "#0288d1")
    record_file = record(sec=6)

    # Passo 2: Transcrever
    result = model_whisper.transcribe(record_file, fp16=False, language=language)
    transcription = result["text"]
    exibir_card("Você perguntou", transcription, "#555")

    # Passo 3: Pensar (Gemini)
    response = brio.generate_content(transcription)
    brio_text = response.text
    exibir_card("Brio respondeu", brio_text, "#d32f2f")

    # Passo 4: Falar (gTTS)
    gtts_object = gTTS(text=brio_text, lang=language, slow=False)
    response_audio = "/content/response_audio.wav"
    gtts_object.save(response_audio)

    display(HTML("<p style='font-size: 18px; color: #d32f2f; font-weight: bold; margin-left: 20px;'>🔊 Ouvindo resposta sonora...</p>"))
    display(Audio(response_audio, autoplay=True))

except Exception as e:
    exibir_card("ERRO TÉCNICO", f"Ocorreu um problema: {str(e)}", "orange")

Output hidden; open in https://colab.research.google.com to view.